# Notebook 04 — Chunking Strategies

---

## Why You Cannot Embed a Whole Document

Embedding models accept a limited amount of text at a time.
The `all-MiniLM-L6-v2` model handles up to 256 tokens (roughly 180-200 words).
Larger models handle up to 8192 tokens, but still have a ceiling.

More importantly, even if the model could accept 50 pages, the resulting embedding
would be a blurry average of everything in those pages. A query about price objections
would match a 50-page document that mentions price objections once, on page 42,
and also covers 30 other topics. That is not useful retrieval.

The solution is to split the document into smaller pieces before embedding.
Each piece gets its own embedding. When you search, you retrieve the specific
piece that answers the question, not the entire document.

---

## The Three Strategies

**Fixed-size** — split at every N characters, regardless of sentence boundaries.
Simple. Can cut mid-sentence. Add overlap to reduce boundary damage.

**Sentence-based** — split at sentence endings (period, question mark, exclamation mark).
Preserves complete thoughts. Better for prose.

**Paragraph-based** — split at blank lines.
Preserves the document's natural structure. Best when each paragraph covers one topic.

---

## The Role of Overlap

For fixed-size chunking, overlap is the number of characters shared between
consecutive chunks. Without overlap, a sentence that falls exactly at a boundary
gets cut in half — the two halves end up in different chunks and neither half
carries the full meaning.

With overlap, the tail of one chunk becomes the head of the next.
Both chunks contain the complete boundary sentence.

Typical overlap: 10 to 20 percent of chunk_size.

---

In [ ]:
import sys
sys.path.append('..')

from utils.chunker import fixed_size_chunks, sentence_chunks, paragraph_chunks

In [ ]:
# Load the sample document
with open('../data/sample_sales_notes.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Document loaded. Total characters: {len(text)}")
print()
print("First 300 characters:")
print(text[:300])

---

## Strategy 1 — Fixed-Size Chunking

In [ ]:
chunks_fixed = fixed_size_chunks(text, chunk_size=500, overlap=50)

print(f"Strategy: fixed-size | chunk_size=500 | overlap=50")
print(f"Number of chunks: {len(chunks_fixed)}")
print()
for i, chunk in enumerate(chunks_fixed[:3]):
    print(f"--- Chunk {i + 1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

In [ ]:
# Look at a boundary — does it cut mid-sentence?
# Check the last 100 chars of chunk 1 and first 100 of chunk 2
print("End of chunk 1:", repr(chunks_fixed[0][-100:]))
print()
print("Start of chunk 2:", repr(chunks_fixed[1][:100]))

---

## Strategy 2 — Sentence-Based Chunking

In [ ]:
chunks_sentence = sentence_chunks(text, max_size=500)

print(f"Strategy: sentence-based | max_size=500")
print(f"Number of chunks: {len(chunks_sentence)}")
print()
for i, chunk in enumerate(chunks_sentence[:3]):
    print(f"--- Chunk {i + 1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

In [ ]:
# Check that sentence boundaries are clean
print("End of chunk 1:", repr(chunks_sentence[0][-60:]))
print()
print("Start of chunk 2:", repr(chunks_sentence[1][:60]))

---

## Strategy 3 — Paragraph-Based Chunking

In [ ]:
chunks_paragraph = paragraph_chunks(text, max_size=800)

print(f"Strategy: paragraph-based | max_size=800")
print(f"Number of chunks: {len(chunks_paragraph)}")
print()
for i, chunk in enumerate(chunks_paragraph[:3]):
    print(f"--- Chunk {i + 1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

---

## Comparing All Three Strategies

In [ ]:
strategies = [
    ("Fixed-size (500, overlap 50)", chunks_fixed),
    ("Sentence (max 500)",           chunks_sentence),
    ("Paragraph (max 800)",          chunks_paragraph),
]

print(f"{'Strategy':<35} {'Chunks':>7} {'Avg size':>10} {'Min':>7} {'Max':>7}")
print("-" * 68)

for name, chunks in strategies:
    sizes = [len(c) for c in chunks]
    avg = sum(sizes) / len(sizes)
    print(f"{name:<35} {len(chunks):>7} {avg:>10.0f} {min(sizes):>7} {max(sizes):>7}")

---

## Effect on Retrieval Quality

Let's embed the chunks from all three strategies, build a FAISS index for each,
and run the same query to see which strategy returns better results.

In [ ]:
from utils.embedder import embed, embed_batch
from utils.searcher import build_faiss_index, faiss_search

query = "how to respond when a prospect says the price is too high"
query_vector = embed(query)

print("Query:", query)
print()

for strategy_name, chunks in strategies:
    # Embed all chunks for this strategy
    chunk_vectors = embed_batch(chunks)
    index = build_faiss_index(chunk_vectors)
    
    # Search for top 1 result
    dists, idxs = faiss_search(index, query_vector, k=1)
    best_chunk = chunks[idxs[0]]
    
    print(f"Strategy: {strategy_name}")
    print(f"Distance: {dists[0]:.4f}")
    print(f"Result  : {best_chunk[:200]}..." if len(best_chunk) > 200 else f"Result: {best_chunk}")
    print()

---

## Choosing Chunk Size

There is no universally correct chunk size. It depends on your data and your queries.

- Smaller chunks (200-400 chars) retrieve more precise passages but less context.
- Larger chunks (600-1500 chars) give more context per result but match less precisely.
- If your LLM will generate an answer from the retrieved chunks, it needs enough context.
  Too-small chunks may not contain all the information needed for a good answer.

A practical starting point: 500 characters, 50 character overlap, sentence-aware.
Adjust based on what your retrieved results actually look like.

---

## Summary

- Documents must be split before embedding. Embedding a whole document produces poor retrieval.
- Fixed-size is simple. Add overlap to reduce boundary damage.
- Sentence-based preserves complete thoughts. Better for prose.
- Paragraph-based preserves document structure. Best for structured content.
- The best strategy for your use case depends on your data — test all three.

Next: Notebook 05 puts it all together into a working RAG pipeline.